#Ingest the circuits csv file
1. Read the file using spark DataframeReader API
2. Add metadata columns 
   -Source File
   -Ingestion Timestamp
3. Write to Bronze Delta Tables


### Read the CSV using Spark Dataframe API


In [0]:
%run ../00_common/01_environment_config

In [0]:
%run ../00_common/02_bronze_helpers

In [0]:
source_file= f"{landing_folder_path}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema= StructType([
            StructField("circuitId", StringType()),
            StructField("url", StringType()),
            StructField("circuitName", StringType()),
            StructField("lat", DoubleType()),
            StructField("long", DoubleType()),
            StructField("locality", StringType()),
            StructField("country",StringType())
            ])

In [0]:
circuits_df=(
    spark.read
    .format("csv")
    .option("header","True")
    #.option("inferSchema", "True")
    .option("mode", "FAILFAST")
    .schema(circuits_schema)
    .load(source_file))


In [0]:
display(circuits_df)

### Adding 2 new colums
- Source File
- Ingestion Timestamp


In [0]:

circuits_final_df = new_columns_insertion(circuits_df)
display(circuits_final_df)

In [0]:
(circuits_final_df
.write
.format("delta")
.mode("overwrite")
.saveAsTable(table_name))

In [0]:
%sql
SELECT * FROM formula1.bronze.circuits

In [0]:
display(spark.table(table_name))